# Comparing Steady State ρ with Theory

This notebook compares the steady state opinion density $\rho$ from simulation with the theoretical prediction for voter model dynamics with stochastic resetting on RRG and ER networks.

In [ ]:
project_root = isdir(joinpath(pwd(), "src")) ? pwd() : dirname(dirname(pwd()))
include(joinpath(project_root, "src", "VoterResetting.jl"))
using .VoterResetting
using Statistics, Plots, Printf, LinearAlgebra, Random, Graphs

# Parameters
N = 1000
μ_rrg_values = [3, 4, 5, 6]  # Whole degrees for RRG
r_values = LinRange(0, 20, 5)  # 5 r-values for reasonable computation time
nsamples = 200
T_steady = 200

fig_dir = joinpath(project_root, "figures", "complex")
mkpath(fig_dir)

println("Setup complete: N=$N, μ=$(μ_rrg_values), nsamples=$nsamples")

# Theoretical Reference

For an RRG with degree $\mu$, the theoretical steady-state ensemble average at the mean-field level (for $\sigma_+ = 1/2$) provides a reference value. 

Key reference: For the voter model without resetting on a $k$-regular graph:
$$\rho^* = \frac{2(k-2)}{k-1}$$

With resetting ($r > 0$), the density typically decreases as the system is periodically reset to a homogeneous state.

In [11]:
function theoretical_rho_reference(μ::Int)
    """
    Reference steady-state opinion density for RRG with degree μ
    at the mean-field level (no resetting, σ+ = 1/2).
    """
    if μ <= 2
        return NaN
    end
    return 2.0 * (μ - 2) / (μ - 1)
end

function theoretical_rho_star(μ::Float64, σ_plus::Float64, r::Float64, ρ_0::Float64; branch::Symbol=:positive)
    """
    Compute theoretical steady state density ρ* from the full formula.
    
    Parameters:
    - μ: mean degree
    - σ_plus: probability/fraction for majority state
    - r: resetting rate
    - ρ_0: reset state density
    - branch: :positive or :negative for the ± in the formula
    
    Returns:
    - ρ*: steady state density (or NaN if formula is undefined)
    """
    
    # Avoid division by zero
    if σ_plus <= 0 || σ_plus >= 1 || μ <= 1
        return NaN
    end
    
    # First parenthesis term
    paren_1 = (μ - 1) / (μ * σ_plus * (1 - σ_plus))
    
    # Inside the square root and outer parenthesis
    coeff = (2 * (μ - 2) / μ - r)
    
    # Term under square root
    discriminant = coeff^2 + 4 * paren_1 * r * ρ_0
    
    # Avoid complex roots
    if discriminant < 0
        return NaN
    end
    
    # Compute the two solutions
    sqrt_term = sqrt(discriminant)
    if branch == :positive
        numerator = coeff + sqrt_term
    else
        numerator = coeff - sqrt_term
    end
    
    # Final formula: (1/2) * paren_1^(-1) * numerator
    rho_star = 0.5 * (1 / paren_1) * numerator
    
    return rho_star
end

println("Theoretical formula implemented")

Theoretical formula implemented


# Simulate Steady State with Resetting

Run simulations on RRG and ER networks to extract the steady state opinion density $\rho$ for various resetting rates and network parameters.

In [12]:
function run_simulation_rrg(N::Int, k::Int, r::Float64, T::Int, n_samples::Int)
    """
    Run ensemble simulation on RRG network with degree k and resetting rate r.
    Returns: (ρ_avg, σ_plus_avg) - ensemble averages at steady state
    """
    # Create RRG graph
    G = random_regular_graph(N, k)
    
    # Setup parameters
    m0_val = 0.0
    delta_reset_protocol = VoterResetting.delta_reset(m0_val)
    params = VoterResetting.ComplexParams(r, m0_val)
    
    # Run ensemble simulation to extract steady state
    times_test = [float(T)]
    result = VoterResetting.simulate_degree_evolution_complex(
        G,
        params;
        reset=delta_reset_protocol,
        times=times_test,
        nsamples=n_samples,
        mode=:all,
    )
    
    # Extract opinion density at steady state
    k_idx = findfirst(==(k), result.k_values)
    if k_idx === nothing
        return NaN, NaN
    end
    
    s_final = result.s_values[k_idx][1, :]  # Final time, all m values
    i_final = result.i_values[k_idx][1, :]
    
    m_vals = collect(0:k)
    km_vals = k .- m_vals
    
    # Compute opinion density ρ: average of two estimators
    sigma_plus = mean(i_final)  # Fraction in state +1
    rho_from_s = mean(s_final .* m_vals) / k
    rho_from_i = mean(i_final .* km_vals) / k
    rho_active = 0.5 * (rho_from_s + rho_from_i)
    
    return rho_active, sigma_plus
end

println("Simulation functions defined")

Simulation functions defined


# Compare Simulation Results with Theory

Run simulations over a grid of (r, μ) parameter space and compare with theoretical predictions.

In [ ]:
# Complete pipeline: Data generation + Visualization for RRG and ER

function theoretical_rho_star(μ::Float64, σ_plus::Float64, r::Float64, ρ_0::Float64)
    """Full theoretical formula with resetting (positive branch)."""
    if σ_plus <= 0 || σ_plus >= 1 || μ <= 1
        return NaN
    end

    paren_1 = (μ - 1) / (μ * σ_plus * (1 - σ_plus))
    coeff = 2 * (μ - 2) / μ - r
    discriminant = coeff^2 + 4 * paren_1 * r * ρ_0
    discriminant < 0 && return NaN

    sqrt_term = sqrt(discriminant)
    numerator = coeff + sqrt_term
    return 0.5 * (1.0 / paren_1) * numerator
end

function build_reset_protocol(kind::Symbol, m0::Float64)
    if kind == :delta
        return VoterResetting.delta_reset(m0)
    elseif kind == :random_node
        return VoterResetting.random_node_reset(m0)
    elseif kind == :hub_highest
        return VoterResetting.hub_reset(m0; highest=true)
    elseif kind == :hub_lowest
        return VoterResetting.hub_reset(m0; highest=false)
    else
        error("Unknown reset protocol kind: $kind")
    end
end

function draw_reset_state(G, protocol)
    N = nv(G)
    state = fill(Int8(-1), N)

    if protocol isa VoterResetting.DeltaReset
        n_plus = Int(round(N * (1 + protocol.target_magnetization) / 2))
        order = randperm(N)
        for i in 1:n_plus
            state[order[i]] = Int8(1)
        end
    elseif protocol isa VoterResetting.RandomNodeReset
        p = 0.5 * (1 + protocol.target_magnetization)
        for i in 1:N
            state[i] = rand() < p ? Int8(1) : Int8(-1)
        end
    elseif protocol isa VoterResetting.HubReset
        deg = degree(G)
        order = sortperm(deg; rev=protocol.highest_first)
        n_plus = Int(round(N * (1 + protocol.target_magnetization) / 2))
        for i in 1:n_plus
            state[order[i]] = Int8(1)
        end
    elseif protocol isa VoterResetting.StateVectorReset
        state .= protocol.state
    else
        error("Reset protocol not supported in rho0 estimator: $(typeof(protocol))")
    end

    return state
end

function active_link_density_from_state(G, state::Vector{Int8})
    ne(G) == 0 && return NaN
    active = 0
    for e in edges(G)
        u = src(e)
        v = dst(e)
        if state[u] != state[v]
            active += 1
        end
    end
    return active / ne(G)
end

function estimate_reset_rho0(G, protocol; nsamples::Int=200)
    if protocol isa VoterResetting.HubReset || protocol isa VoterResetting.StateVectorReset
        st = draw_reset_state(G, protocol)
        return active_link_density_from_state(G, st)
    end

    vals = Vector{Float64}(undef, nsamples)
    for i in 1:nsamples
        st = draw_reset_state(G, protocol)
        vals[i] = active_link_density_from_state(G, st)
    end
    return mean(vals)
end

function extract_global_rho_sigma(result)
    # Aggregate over all degree classes present in the graph.
    # Skip k=0 in rho terms and guard against any non-finite class entries.
    sum_i = 0.0
    rho_from_s = 0.0
    rho_from_i = 0.0

    for (idx, k) in enumerate(result.k_values)
        s_row = result.s_values[idx][1, :]
        i_row = result.i_values[idx][1, :]

        if all(isfinite.(i_row))
            sum_i += sum(i_row)
        end

        if k > 0 && all(isfinite.(s_row)) && all(isfinite.(i_row))
            m_vals = collect(0:k)
            rho_from_s += dot(s_row, m_vals) / k
            rho_from_i += dot(i_row, k .- m_vals) / k
        end
    end

    rho = 0.5 * (rho_from_s + rho_from_i)
    sigma_plus = sum_i
    return rho, sigma_plus
end

function build_er_without_isolates(N::Int, p::Float64; max_tries::Int=30)
    for _ in 1:max_tries
        G = erdos_renyi(N, p)
        if minimum(degree(G)) > 0
            return G
        end
    end
    # Fallback if no-isolates graph is not found quickly.
    return erdos_renyi(N, p)
end

function run_sim_on_graph(G, r::Float64, T::Int, n_samp::Int, reset_protocol)
    params = VoterResetting.ComplexParams(r, 0.0)
    rho0_reset = estimate_reset_rho0(G, reset_protocol; nsamples=150)

    result = VoterResetting.simulate_degree_evolution_complex(
        G, params;
        reset=reset_protocol,
        times=[float(T)],
        nsamples=n_samp,
        mode=:all,
    )

    rho, sigma_plus = extract_global_rho_sigma(result)
    return rho, sigma_plus, rho0_reset
end

# Explicit protocol choice for complex-graph resetting.
# Options: :delta, :random_node, :hub_highest, :hub_lowest
reset_kind = :random_node
reset_m0 = 0.0
reset_protocol = build_reset_protocol(reset_kind, reset_m0)
println("Reset protocol kind: $reset_kind")
println("Reset protocol object: $(typeof(reset_protocol))")

println("Generating data for RRG and ER... ")
rho_rrg = Dict{Int, Vector{Float64}}()
sigma_rrg = Dict{Int, Vector{Float64}}()
rho_theory_rrg = Dict{Int, Vector{Float64}}()

rho_er = Dict{Int, Vector{Float64}}()
sigma_er = Dict{Int, Vector{Float64}}()
rho_theory_er = Dict{Int, Vector{Float64}}()

for μ in μ_rrg_values
    rho_rrg[μ] = zeros(length(r_values))
    sigma_rrg[μ] = zeros(length(r_values))
    rho_theory_rrg[μ] = zeros(length(r_values))

    rho_er[μ] = zeros(length(r_values))
    sigma_er[μ] = zeros(length(r_values))
    rho_theory_er[μ] = zeros(length(r_values))

    for (i, r) in enumerate(r_values)
        # RRG with exact degree μ
        G_rrg = random_regular_graph(N, μ)
        rho_s_rrg, σ_rrg, rho0_rrg = run_sim_on_graph(G_rrg, r, T_steady, nsamples, reset_protocol)
        rho_rrg[μ][i] = rho_s_rrg
        sigma_rrg[μ][i] = σ_rrg
        rho_theory_rrg[μ][i] = theoretical_rho_star(float(μ), σ_rrg, r, rho0_rrg)

        # ER with mean degree approximately μ; avoid isolates for numerical stability.
        p_er = μ / (N - 1)
        G_er = build_er_without_isolates(N, p_er)
        rho_s_er, σ_er, rho0_er = run_sim_on_graph(G_er, r, T_steady, nsamples, reset_protocol)
        rho_er[μ][i] = rho_s_er
        sigma_er[μ][i] = σ_er
        rho_theory_er[μ][i] = theoretical_rho_star(float(μ), σ_er, r, rho0_er)

        @printf("μ=%d, r=%5.1f | RRG: ρ=%.4f, σ+=%.4f, ρ0=%.4f, ρ_th=%.4f | ER: ρ=%.4f, σ+=%.4f, ρ0=%.4f, ρ_th=%.4f\n",
            μ, r,
            rho_s_rrg, σ_rrg, rho0_rrg, rho_theory_rrg[μ][i],
            rho_s_er, σ_er, rho0_er, rho_theory_er[μ][i])
    end
end

println("\n" * "="^60)
println("Creating visualizations...")
println("="^60 * "\n")

colors = [:dodgerblue, :crimson, :olive, :darkcyan]
markers = [:circle, :square, :diamond, :star5]

# RRG combined plot
p_rrg = plot(
    xlabel="r (resetting rate)",
    ylabel="ρ  (ensemble avg)",
    title="RRG: Sim vs Theory (protocol=$(reset_kind))",
    size=(1050, 650),
    legend=:outerright,
    margin=5Plots.mm
)

for (idx, μ) in enumerate(μ_rrg_values)
    plot!(p_rrg, r_values, rho_rrg[μ];
        lw=2.5, marker=markers[idx], markersize=7, color=colors[idx],
        label="RRG sim μ=$μ")
    plot!(p_rrg, r_values, rho_theory_rrg[μ];
        lw=2, ls=:dash, color=colors[idx],
        label="RRG th μ=$μ")
end
display(p_rrg)
savefig(p_rrg, joinpath(fig_dir, "rho_vs_r_rrg_sim_theory.png"))
println("Saved: rho_vs_r_rrg_sim_theory.png")

# ER combined plot
p_er = plot(
    xlabel="r (resetting rate)",
    ylabel="ρ  (ensemble avg)",
    title="ER: Sim vs Theory (protocol=$(reset_kind))",
    size=(1050, 650),
    legend=:outerright,
    margin=5Plots.mm
)

for (idx, μ) in enumerate(μ_rrg_values)
    plot!(p_er, r_values, rho_er[μ];
        lw=2.5, marker=markers[idx], markersize=7, color=colors[idx],
        label="ER sim ⟨k⟩≈$μ")
    plot!(p_er, r_values, rho_theory_er[μ];
        lw=2, ls=:dash, color=colors[idx],
        label="ER th ⟨k⟩≈$μ")
end
display(p_er)
savefig(p_er, joinpath(fig_dir, "rho_vs_r_er_sim_theory.png"))
println("Saved: rho_vs_r_er_sim_theory.png")

# Side-by-side quick comparison
p_both = plot(p_rrg, p_er; layout=(1,2), size=(1800, 700))
display(p_both)
savefig(p_both, joinpath(fig_dir, "rho_vs_r_rrg_er_comparison.png"))
println("Saved: rho_vs_r_rrg_er_comparison.png")

println("="^60)
println("Visualization complete!")

Reset protocol kind: random_node
Reset protocol object: Main.VoterResetting.RandomNodeReset
Generating data for RRG and ER... 
μ=3, r=  0.0 | RRG: ρ=0.0685, σ+=0.4944, ρ0=0.4989, ρ_th=0.2500 | ER: ρ=NaN, σ+=0.4752, ρ0=0.5004, ρ_th=0.2494
μ=3, r=  5.0 | RRG: ρ=0.0876, σ+=0.5008, ρ0=0.5014, ρ_th=0.4525 | ER: ρ=NaN, σ+=0.5020, ρ0=0.4991, ρ_th=0.4508
μ=3, r= 10.0 | RRG: ρ=0.0987, σ+=0.5229, ρ0=0.5008, ρ_th=0.4726 | ER: ρ=NaN, σ+=0.4735, ρ0=0.5006, ρ_th=0.4724
μ=3, r= 15.0 | RRG: ρ=0.1086, σ+=0.4884, ρ0=0.5007, ρ_th=0.4809 | ER: ρ=NaN, σ+=0.4887, ρ0=0.5003, ρ_th=0.4805
μ=3, r= 20.0 | RRG: ρ=0.1114, σ+=0.5119, ρ0=0.4999, ρ_th=0.4847 | ER: ρ=NaN, σ+=0.5227, ρ0=0.4996, ρ_th=0.4844
μ=4, r=  0.0 | RRG: ρ=0.0499, σ+=0.4304, ρ0=0.5005, ρ_th=0.3269 | ER: ρ=NaN, σ+=0.4884, ρ0=0.5009, ρ_th=0.3332


# Additional Diagnostics

The main RRG + ER simulation/theory figures are generated in the previous code cell.

This section provides lightweight summary diagnostics from those computed arrays.

In [ ]:
# Style-matched plots: simulation traces + dashed theory + horizontal reference lines

colors = Dict(3 => :dodgerblue, 4 => :crimson, 5 => :olive, 6 => :darkcyan)
markers = Dict(3 => :circle, 4 => :square, 5 => :diamond, 6 => :star5)

# The r=0 regular-graph reference often used as a guide line.
ame_reference(mu::Int) = 2.0 * (mu - 2) / (mu - 1)

p_style_rrg = plot(
    size=(1000, 620),
    xlabel="r",
    ylabel="ρ",
    title="RRG: simulation + theory + horizontal reference",
    legend=:outerright,
    margin=5Plots.mm,
)

for μ in μ_rrg_values
    y_sim = rho_rrg[μ]
    y_theory = rho_theory_rrg[μ]
    y_ref = ame_reference(μ)

    plot!(p_style_rrg, r_values, y_sim;
        lw=2.2, marker=markers[μ], markersize=6, alpha=0.8,
        color=colors[μ], label="sim μ=$(μ)")
    plot!(p_style_rrg, r_values, y_theory;
        lw=2.2, ls=:dash, color=colors[μ], label="theory μ=$(μ)")
    hline!(p_style_rrg, [y_ref];
        lw=1.2, ls=:dot, color=colors[μ], alpha=0.65, label="")
end

display(p_style_rrg)
savefig(p_style_rrg, joinpath(fig_dir, "rho_style_rrg_sim_theory_ref.png"))
println("Saved: rho_style_rrg_sim_theory_ref.png")

p_style_er = plot(
    size=(1000, 620),
    xlabel="r",
    ylabel="ρ",
    title="ER: simulation + theory + horizontal reference",
    legend=:outerright,
    margin=5Plots.mm,
)

for μ in μ_rrg_values
    y_sim = rho_er[μ]
    y_theory = rho_theory_er[μ]
    y_ref = ame_reference(μ)

    plot!(p_style_er, r_values, y_sim;
        lw=2.2, marker=markers[μ], markersize=6, alpha=0.8,
        color=colors[μ], label="sim ⟨k⟩≈$(μ)")
    plot!(p_style_er, r_values, y_theory;
        lw=2.2, ls=:dash, color=colors[μ], label="theory ⟨k⟩≈$(μ)")
    hline!(p_style_er, [y_ref];
        lw=1.2, ls=:dot, color=colors[μ], alpha=0.65, label="")
end

display(p_style_er)
savefig(p_style_er, joinpath(fig_dir, "rho_style_er_sim_theory_ref.png"))
println("Saved: rho_style_er_sim_theory_ref.png")

p_style_both = plot(p_style_rrg, p_style_er; layout=(1,2), size=(1900, 680))
display(p_style_both)
savefig(p_style_both, joinpath(fig_dir, "rho_style_rrg_er_sim_theory_ref.png"))
println("Saved: rho_style_rrg_er_sim_theory_ref.png")

In [ ]:
# Summary statistics (RRG and ER)

println("=" ^ 60)
println("Rho comparison summary")
println("=" ^ 60)

for μ in μ_rrg_values
    rho_rrg_valid = rho_rrg[μ][isfinite.(rho_rrg[μ])]
    rho_er_valid = rho_er[μ][isfinite.(rho_er[μ])]

    @printf("\nμ=%d\n", μ)
    if !isempty(rho_rrg_valid)
        @printf("  RRG: mean=%.4f, std=%.4f, range=[%.4f, %.4f]\n",
            mean(rho_rrg_valid), std(rho_rrg_valid), minimum(rho_rrg_valid), maximum(rho_rrg_valid))
    end
    if !isempty(rho_er_valid)
        @printf("  ER : mean=%.4f, std=%.4f, range=[%.4f, %.4f]\n",
            mean(rho_er_valid), std(rho_er_valid), minimum(rho_er_valid), maximum(rho_er_valid))
    end
end

println("\n" * "=" ^ 60)